# The Evolution of the Gradient Voting Algorithm

This notebook documents the experimental process of tuning the custom gradient-based voting algorithm. Over the course of 28 iterations (`A_1` to `A_28`), various hyperparameters were tested, including blur types, kernel sizes, search grid densities, and threshold values.

In [ ]:
import sys
from pathlib import Path
sys.path.append('..')  # Add parent directory to path to import src

from src.data import load_dataset_paths, load_ground_truths
from src.visualization import show_results

# Import the historical iterations
from src.algorithms.gradient_experiments import (
    find_circle_A_baseline,
    find_circle_A_11_median,
    find_circle_A_15_radius,
    find_circle_A_27_dense
)
# Import the final, highly optimized version (A_28)
from src.algorithms.gradient import find_circle_gradient

img_dir = Path('../data/circle')
train_set, val_set = load_dataset_paths(img_dir)
ground_truths = load_ground_truths(img_dir)
print(f"Loaded validation set with {len(val_set)} images.")

### 1. The Baseline Version (`A_1` to `A_7` era)
Initial testing with basic Gaussian Blur, `big_step=7`, and a small threshold. It worked well but struggled with noisy edges.

In [ ]:
show_results(val_set, img_dir, ground_truths, algo=find_circle_A_baseline)

### 2. Iteration `A_11`: Median Blur
Here, we tested replacing GaussianBlur with MedianBlur to see if it handled sharp reflections on the cornea better.

In [ ]:
show_results(val_set, img_dir, ground_truths, algo=find_circle_A_11_median)

### 3. Iteration `A_15`: Increased Search Radius & Thresholding
We increased the `search_radius` for Phase 2 from 6 to 15, and bumped the gradient threshold to 5.0. This significantly reduced false positives caused by faint skin textures, and we used a strict $cos^3(\alpha)$ penalty for misaligned gradient vectors.

In [ ]:
show_results(val_set, img_dir, ground_truths, algo=find_circle_A_15_radius)

### 4. Iteration `A_27`: High Density Search
In this phase, we tested brute computational power. We increased `n_points` from 64 to 128 in Phase 1, and up to 512 in Phase 2. This was highly accurate but very slow.

In [ ]:
show_results(val_set, img_dir, ground_truths, algo=find_circle_A_27_dense)

### 5. The Final Version: `A_28`
The sweet spot! We settled on `blur_kernel_size=21` (Gaussian), `big_step=6`, `search_radius=15`, and a very strict $cos^5(\alpha)$ vector alignment penalty. We reverted `n_points` to 64 and 128 to keep execution time under control while maintaining maximum accuracy.

In [ ]:
show_results(val_set, img_dir, ground_truths, algo=find_circle_gradient)